In [ ]:
import pandas as pd
import re
import ast

#used pydantic 2.9.2
from pydantic import BaseModel
#used openai 1.59.2
from openai import OpenAI

import os

In [ ]:
#These three strings are how the LLM was instructed to transform the data
#These can be changed and tested in different variations
#Probably easier to read these long strings if you copy to a text editor
songs_system_prompt='You will be given a portion of a transcript from a video that has been translated and may contain errors. You must decide if there are any songs in the transcript. Output a python list of the songs and a python list of the artists. Both lists must be the same length. Output two empty lists if there are no songs.'
verses_system_prompt='You will be given a portion of a transcript from a video that has been translated and may contain errors. You must decide if there are any Bible verses quoted either directly or indirectly in the transcript. Output a python list of Bible verses in the form ["2 Kings 3:3-5", "John 2:8"]. Return an empty list if there are no verses. For each verse output you also will provide a score from 0 to 100 indicating how directly the verse is quoted, a full direct quote would score 100.'
prayer_system_prompt='You will be given a portion of a transcript from a video that has been translated and may contain errors. You must decide if there are any prayers in this section of the transcript. All prayers must end with the word amen. If you identify a prayer, directly quote the prayer and output as a python list. Output an empty list if there are no prayers.'

In [ ]:
#This script uses chat gpt-4o
#Use your openAI API key or change script to run with your choice of LLM
api_key = "api_key_here"


In [ ]:
#This function removes punctuation and timestamps from the transcript 
def clean_text(df):
    list1=[]
    for x in df['Text']:
        list1.append(re.sub(r'[^a-zA-Z., ]', '', x.split('] ')[1]))
    length=int(len(list1)/10)

    text_list=[]
    for i in range(0,10):
        temp_text=''
        if i<9:
            for t in list1[i*length:(i+1)*length+50]:
                temp_text+=t
        else:
            for t in list1[i*length:]:
                temp_text+=t   
        temp_text=temp_text.replace('.','. ')
        temp_text=temp_text.replace('  ',' ')
        text_list.append(temp_text)
    return text_list

#The following classes are used to get a structed output out of the LLM
class output_songs(BaseModel):
    song_name: list[str]
    artist: list[str]

class output_verses(BaseModel):
    verses: list[str]
    score: list[str]

class output_prayers(BaseModel):
    prayer: list[str]

#This is the call to OpenAI API function
def query_model(system_prompt,user_prompt,output_format):

    client = OpenAI()

    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=output_format,
    )

    return completion.choices[0].message.parsed

In [ ]:
#this script is designed to grab all of the transcript files you parsed in the previous notebook
#whatever file you sent the transcripts to can be called here

folders=[]
files=[]

for folder in os.listdir("~filepath"):
    if '.' in folder:
        pass
    else:
        folders.append("~filepath"+str(folder))

for folder in folders:
    for file in os.listdir(folder):
        if 'transcribe' in str(file):
            files.append(folder+'\\'+file)



In [ ]:
#this is the cell that runs your repeated calls to the LLM
#It is designed to run until it breaks and then log results up to that point
#For this example the LLM would throws occasional errors after hundreds of queries but work after reset

file_name=[]
file_songs=[]
file_verses=[]
file_prayers=[]

total_files_len=len(files)
#This starts at -1 but if the script ever stops you should update count to the number where the script stopped
count=-1

for f in files[count+1:]:
    count+=1
    print('working on file '+str(count)+' of '+str(total_files_len))
    file_name.append(f)
    df=pd.read_csv(f)
    text_list=clean_text(df)

    verse_list=[]
    song_list=[]
    prayer_list=[]

    for text in text_list:
        #the next three lines are the calls to the LLM
        song_list.append(query_model(songs_system_prompt,str(text),output_songs))
        verse_list.append(query_model(verses_system_prompt,str(text),output_verses))
        prayer_list.append(query_model(prayer_system_prompt,str(text),output_prayers))

    file_songs.append(song_list)
    file_verses.append(verse_list)
    file_prayers.append(prayer_list)

    output_df=pd.DataFrame()
    output_df['file_names']=file_name
    output_df['file_songs']=file_songs
    output_df['file_verses']=file_verses
    output_df['file_prayers']=file_prayers

    #The csv gets overwritten every successful run
    #If you get an error and the script stops, update the name of the file below from 1 to 2 etc.
    output_df.to_csv("~filepath\\chat_gpt_results_1.csv")
    


In [ ]:
#once the queries are done we begin to a assemble a single dataframe of results
#The following functions are used to turn the structured output into a form we can use
def parse_output_verses(s):
    # Regular expression to extract content inside output_verses()
    pattern = r"output_verses\(verses=(\[[^\]]*\]), score=(\[[^\]]*\])\)"
    
    # Find all matches
    matches = re.findall(pattern, s)
    
    # Convert extracted strings into Python lists using ast.literal_eval()
    parsed_list = [{"verses": ast.literal_eval(verses), "score": ast.literal_eval(score)} for verses, score in matches]
    
    return parsed_list

def parse_output_songs(s):
    # Regular expression to extract content inside output_songs()
    pattern = r"output_songs\(song_name=(\[[^\]]*\]), artist=(\[[^\]]*\])\)"
    
    # Find all matches
    matches = re.findall(pattern, s)
    
    # Convert extracted strings into Python lists using ast.literal_eval()
    parsed_list = [{"song_name": ast.literal_eval(song_name), "artist": ast.literal_eval(artist)} for song_name, artist in matches]
    
    return parsed_list

def parse_output_prayers(s):
    # Regular expression to extract content inside output_songs()
    pattern = r"output_prayers\(prayer=(\[[^\]]*\])\)"
    # Find all matches
    matches = re.findall(pattern, s)
    
    # Convert extracted strings into Python lists using ast.literal_eval()
    parsed_list = [{"prayer": ast.literal_eval(prayer)} for prayer in matches]
    
    return parsed_list

In [ ]:
#combine chat gpt result files and clean output into one dataframe

list1=[]
#this range should go from 1 to however many results .csv you had to create
for x in range(1,7):
    list1.append(pd.read_csv("~filepath\\chat_gpt_results_"+str(x)+'.csv'))
df=pd.concat(list1,axis=0,ignore_index=True)

list1=[]
files_found=[]
for x in df['file_names']:
    if x in files_found:
        list1.append(1)
    else:
        list1.append(0)
    files_found.append(x)
df['stat1']=list1
df=df[df['stat1']==0]

files=[]

song1=[]
song2=[]
song3=[]
song4=[]
song5=[]
song6=[]
song7=[]
song8=[]
song9=[]
song10=[]

verse1=[]
verse2=[]
verse3=[]
verse4=[]
verse5=[]
verse6=[]
verse7=[]
verse8=[]
verse9=[]
verse10=[]

prayer1=[]
prayer2=[]
prayer3=[]
prayer4=[]
prayer5=[]
prayer6=[]
prayer7=[]
prayer8=[]
prayer9=[]
prayer10=[]

files=[]
songs=[]
verses=[]
prayers=[]

for i in range(0,df.shape[0]):
    a=df.iloc[i]
    files.append(a['file_names'])
    songs.append(parse_output_songs(a['file_songs']))
    verses.append(parse_output_verses(a['file_verses']))
    prayers.append(parse_output_prayers(a['file_prayers']))

output_df=pd.DataFrame()
output_df['files']=files
for i in songs:
    count=0
    for s in [song1,song2,song3,song4,song5,song6,song7,song8,song9,song10]:
        if len(i[count]['song_name'])>0:
            s.append(str(i[count]['song_name']).replace('[','').replace(']','').replace("'",''))
        else:
            s.append('')
        count+=1

#one verse didn't parse correctly - this hopefully doesn't happen to you but this is a way to fix the problem
verses[242].append({'verses': [], 'score': []})

#one prayer didn't parse correctly - this hopefully doesn't happen to you but this is a way to fix the problem
prayers[330].append({'prayer': []})

#clean verses and remove quotes at 70 score or below
#Recall that verses came with a score in their structed output, scores of 70 correspond to the more complete quotes
count1=-1
for i in verses:
    count1+=1
    count2=-1
    for j in i:
        count2+=1
        try:
            verses_temp = j['verses']
            scores_temp = j['score']

            filtered_verses_scores = [(v, s) for v, s in zip(verses_temp, scores_temp) if int(s) > 70]

            filtered_entry = {
                'verses': [v for v, s in filtered_verses_scores],
                'score': [s for v, s in filtered_verses_scores]
            }

            verses[count1][count2]=filtered_entry
        except:
            verses[count1][count2]={'verses': [], 'score': []}

for i in verses:
    count=0
    for s in [verse1,verse2,verse3,verse4,verse5,verse6,verse7,verse8,verse9,verse10]:
        if len(i[count]['verses'])>0:
            s.append(str(i[count]['verses']).replace('[','').replace(']','').replace("'",''))
        else:
            s.append('')
        count+=1

for i in prayers:
    count=0
    for s in [prayer1,prayer2,prayer3,prayer4,prayer5,prayer6,prayer7,prayer8,prayer9,prayer10]:
        if len(i[count]['prayer'])>0:
            s.append(str(i[count]['prayer']).replace('[','').replace(']','').replace("'",''))
        else:
            s.append('')
        count+=1


output_df['song1']=song1
output_df['song2']=song2
output_df['song3']=song3
output_df['song4']=song4
output_df['song5']=song5
output_df['song6']=song6
output_df['song7']=song7
output_df['song8']=song8
output_df['song9']=song9
output_df['song10']=song10


output_df['verse1']=verse1
output_df['verse2']=verse2
output_df['verse3']=verse3
output_df['verse4']=verse4
output_df['verse5']=verse5
output_df['verse6']=verse6
output_df['verse7']=verse7
output_df['verse8']=verse8
output_df['verse9']=verse9
output_df['verse10']=verse10

output_df['prayer1']=prayer1
output_df['prayer2']=prayer2
output_df['prayer3']=prayer3
output_df['prayer4']=prayer4
output_df['prayer5']=prayer5
output_df['prayer6']=prayer6
output_df['prayer7']=prayer7
output_df['prayer8']=prayer8
output_df['prayer9']=prayer9
output_df['prayer10']=prayer10

output_df.head()
#this is the desired final results file combined
output_df.to_csv("~filepath\\structured_output1.csv")
